# Sepsis Paper Analysis Pipeline

This notebook uses OpenRouter's LLM API to systematically extract and analyze hierarchical claims, evidence, and metadata from sepsis/septic shock research papers. 

It executes a sequence of 6 prompts for each paper to build a comprehensive logical chain, ultimately outputting a structured RDF graph.

In [ ]:
# Install dependencies if needed
!pip install ollama 


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement openrouter_adapter (from versions: none)

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for openrouter_adapter


In [9]:
!pip install --upgrade pip

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.0
    Uninstalling pip-26.0:
      Successfully uninstalled pip-26.0


In [11]:
!pip install openrouter_adapter

ERROR: Could not find a version that satisfies the requirement openrouter_adapter (from versions: none)
ERROR: No matching distribution found for openrouter_adapter


In [6]:
from ollama import chat

## 1. Setup and Configuration

In [ ]:
API_KEY = os.environ.get("OPENROUTER_API_KEY", "sk-or-v1-REPLACE_WITH_YOUR_KEY")

## 1. Initialize Client
Initialize the OpenRouter API client to interact with the chosen LLM.

In [4]:
client = OpenRouterClient(api_key=API_KEY)

NameError: name 'OpenRouterClient' is not defined

## 2. Define Analysis Prompts
Define the 6 distinct prompts that will be fed to the model sequentially for each paper:
1. Core attributes & metadata
2. Hierarchical claims
3. Supporting evidence
4. Challenging/against evidence
5. Logical chain & conflict mapping
6. Turtle syntax RDF graph generation

In [12]:
prompts = [
    """1.With the target paper focused on sepsis/septic shock, first extract and label its core attributes to establish a foundational understanding. Address the following requirements comprehensively:
Classify the paper type by selecting the most accurate option: □ Original Research (specify sub-type: Systematic Review / Meta-Analysis / Cohort Study / Randomized Controlled Trial) □ Commentary / Brief Report □ Consensus Definition. If the paper integrates multiple types, note all relevant categories.
Define the study population clearly: □ Adults (specify age range if provided) □ Children (specify sub-groups: neonates [0–30 days] / infants / toddlers / school-age children / adolescents; include gestational age if relevant) □ Mixed Population (detail the composition of different groups).
Summarize the core research objective in one concise sentence. Ensure it captures the paper’s primary purpose, such as "To revise the clinical criteria for adult septic shock," "To identify mortality-related factors in pediatric sepsis," or "To critique the limitations of the Phoenix criteria for pediatric sepsis."
Supplement key contextual metadata: (1) Publication year and journal; (2)Primary data source (e.g., specific databases: SSC / UPMC / KPNC; electronic health records; registry data).""",
    
    """2.Conduct a thorough analysis of the target sepsis/septic shock paper to extract all hierarchical claims, adhering to the following definitions and requirements:
Definition of a Claim: A clear, testable proposition put forward in the paper, including but not limited to conclusions, definitions, associations, comparisons, criticisms, or recommendations. It must be capable of being supported or refuted by evidence. Exclude descriptive statements (e.g., "The study included 106 articles") that do not express a stance or conclusion.
Extraction Requirements:
Identify the Core Claim: The overarching, most significant conclusion of the paper (e.g., "Septic shock is a subset of sepsis with higher mortality risk due to circulatory, cellular, and metabolic abnormalities" or "Elevated blood lactate is strongly associated with mortality in pediatric sepsis").
Extract Subordinate Claims: All secondary propositions that directly support the core claim (e.g., "A blood lactate cutoff of >2 mmol/L optimally predicts septic shock mortality" or "Pediatric sepsis mortality is higher in low- and middle-income countries than in high-income countries").
Capture Critical Claims (if applicable): For commentaries/brief reports, extract the author’s criticisms of existing standards, definitions, or research (e.g., "The Phoenix criteria fail to adequately incorporate renal and hepatic dysfunction assessments" or "The SIRS criteria lack specificity for pediatric sepsis"). 
For each claim, note its location in the paper (e.g., abstract, results, discussion) and whether it is the author’s original proposition or a cited claim (with attribution if specified).""",

    """3.For each claim extracted from the sepsis/septic shock paper, exhaustively identify and analyze all supporting evidence, following these guidelines:
Definition of Supporting Evidence: Any material in the paper used to validate the truth or validity of a claim, including data, methodological details, literature citations, or logical reasoning. Distinguish clearly between direct evidence (directly confirms the claim) and indirect evidence (supports the claim through inference or analogy).
Extraction Requirements:
For each claim, categorize the evidence type(s) from the following:
□ Data Evidence: Include specific numerical values (sample size, statistics, P-values, odds ratios [OR], mean differences, mortality rates, confidence intervals [95% CI]), e.g., "42.3% mortality (95% CI: 41.2%–43.3%) among 8,520 patients in the SSC dataset" or "Mean lactate level of 4.6 mmol/L in non-survivors vs. 2.7 mmol/L in survivors (P<0.001)".
□ Methodological Evidence: Describe the study design, analytical approaches, or consensus-building processes that support the claim, e.g., "Consensus reached via 3 rounds of the Delphi method involving 19 international experts" or "Validation across three independent datasets (SSC, UPMC, KPNC)".
□ Literature Citation Evidence: Cite relevant studies, guidelines, or consensus reports referenced to support the claim, e.g., "Consistent with the 2023 ADQI consensus on sepsis-associated acute kidney injury" or "Supported by findings from Schlapbach et al. (2024) in JAMA".
□ Logical Deduction Evidence: Explain the reasoning process linking premises to the claim, e.g., "Pupillary light reflex is a late sign of irreversible neurological damage, so it is unsuitable for early sepsis detection".
Assess and mark the strength of the association between each piece of evidence and the corresponding claim: □ Strong (directly validates the claim without additional inference) □ Weak (supports the claim indirectly, e.g., through analogy, correlation, or exclusion of alternative explanations).
For data evidence, fully document the data source: Specify the database name, study period, sample population characteristics, and any exclusion/inclusion criteria relevant to the evidence.
Organize evidence in the order of relevance to the claim (prioritize direct, high-strength evidence first).""",

    """4.Systematically identify and analyze all evidence that challenges, weakens, or limits the claims in the sepsis/septic shock paper, based on the following framework:
Definition of Against Evidence: Any material in the paper that undermines the validity, generalizability, or completeness of a claim, including study limitations, contradictory data, conflicting literature, or logical flaws. This includes both self-acknowledged limitations by the authors and external counter-evidence cited in the paper.
Extraction Requirements:
For each claim, categorize the against evidence type(s) from the following:
□ Limitations of Own Research: Author-identified constraints that affect the claim’s robustness or generalizability, e.g., "LMIC patients accounted for only 1.8% of the sample, limiting extrapolation to low-resource settings" or "Missing data on serum lactate in 15.7% of patients may introduce selection bias".
□ Counter Data/Literature: Data from the paper or cited studies that contradict the claim, e.g., "Some sub-analyses showed no association between bilirubin levels and pediatric sepsis mortality" or "A 2022 study by Saini et al. found that hepatic dysfunction was not an independent predictor of sepsis outcomes".
□ Logical Flaws: Inconsistencies or gaps in the reasoning supporting the claim, e.g., "The Phoenix criteria do not account for fluid resuscitation response, potentially underestimating disease severity" or "The claim relies on cross-sectional data and cannot establish causal relationships".
□ Scope Limitations: Factors that restrict the claim’s applicability to specific populations, settings, or scenarios, e.g., "The proposed criteria are only valid for ICU-admitted patients and not for early sepsis in emergency departments" or "The association between vasoactive agents and mortality may be confounded by disease severity".
Clarify the relationship between each piece of against evidence and the claim: □ Direct Refutation (provides conclusive evidence that the claim is incorrect) □ Scope Restriction (narrows the conditions under which the claim holds true) □ Uncertainty Introduction (raises doubts about the claim’s validity without fully refuting it).
For counter data/literature, include key details such as sample size, statistical results, and publication information to enable traceability.""",

    """5.Construct a clear and comprehensive logical chain by mapping the relationships between claims and evidence in the sepsis/septic shock paper, following these steps:
Develop an association map for each core claim:
Structure: Core Claim ←[Association Strength: Strong/Weak]→ Subordinate Claim 1 ←[Evidence Type: Data/Methodological/Literature/Logical]→ Evidence 1 / Evidence 2; Core Claim ←[Association Strength: Strong/Weak]→ Subordinate Claim 2 ←[Evidence Type: Data/Methodological/Literature/Logical]→ Evidence 3 / Evidence 4.
Example: Core Claim ("Adult septic shock is defined by vasopressor use + lactate >2 mmol/L post-fluid resuscitation") ←[Strong]→ Subordinate Claim ("This combination predicts higher mortality") ←[Data Evidence]→ Evidence ("42.3% mortality in SSC Group 1 vs. 30.1% in Group 2 (P<0.001)").
Identify and document evidence conflicts:
If a claim is supported by both supporting and against evidence, explicitly describe the conflict point, e.g., "Supporting Evidence: Lactate >2 mmol/L is associated with increased mortality; Against Evidence: 15% of septic shock patients had normal lactate levels, leading to potential underdiagnosis".
Analyze the resolution of conflicts (if provided in the paper), such as author explanations, subgroup analyses, or prioritization of stronger evidence.
Evaluate logical closure completeness:
Classify each core claim as: □ Complete (Covers Claim + Supporting Evidence + Against Evidence + Conflict Resolution) □ Incomplete (e.g., No Against Evidence mentioned, Unresolved Conflicts, Insufficient Supporting Evidence).
For incomplete logical closure, note the gaps, e.g., "The paper provides no against evidence for the claim that qSOFA is effective for sepsis screening, limiting assessment of its limitations".
Summarize the overall logical flow: Briefly describe how subordinate claims and evidence collectively support the core claim, and how against evidence or conflicts shape the claim’s final formulation.""",

    """6.Generate a structured, syntactically valid Turtle syntax RDF graph for sepsis/septic shock papers following this strict framework: organize into 4 comment-separated modules (Paper Metadata → Core Claim → Subordinate Claims → Supporting/Against Evidence) with 2-space indentation; use URI naming rules (paper:00X, claim:core00X, claim:sub00X-0X, ev:sup00X-0X-0X, ev:anti00X-0X-0X) and mandatory prefixes (rdfs:, xsd:, sep:, paper:, claim:, ev:); define Paper Metadata (sep:Paper) with rdfs:label, publicationYear, studyPopulation, paperType, coreObjective, hasCoreClaim; Core Claim (sep:CoreClaim) with rdfs:label, claimType (Definition/Association/Comparison/Critical/Recommendation), hasSubClaim; Subordinate Claims (sep:SubClaim) with rdfs:label, claimType, hasSupportingEv, hasAgainstEv (optional); Supporting Evidence (sep:SupportingEvidence) with rdfs:label, evidenceType (Data/Methodological/Literature/Logical), strength (Strong/Weak), relationship (DirectSupport/IndirectSupport), source, statDetails; Against Evidence (sep:AgainstEvidence) with rdfs:label, evidenceType (Data/Limitations/CounterLiterature/LogicalFlaw), relationship (DirectRefutation/ScopeRestriction), source, statDetails (no strength); enforce 1:1 mapping (1 evidence→1 claim, 1 paper→1 core claim, no cross-linking), keep rdfs:label concise (1-2 sentences), use placeholders for dynamic content, and ensure no redundant triples."""
]

## 3. Execute Pipeline and Save Results
Process each paper sequentially. The script loops over all `.txt` files in the input directory, appends the paper content to the first prompt, and dynamically updates the chat history for subsequent prompts. 

Only the final RDF graph result is written to the output directory.

In [13]:
df_input = pd.read_excel(INPUT_FILE)
df_subset = df_input.head(2) 

final_results = []

NameError: name 'INPUT_FILE' is not defined

In [14]:
MODEL_ID = "openai/gpt-5.2-chat" 

for index, row in df_subset.iterrows():
    paper_content = str(row['Content'])
    paper_code = row['Paper_Code']
    model_to_use = row['LLM_Model']
    category = row['Category'] 
    
    messages = []
    final_rdf = ""
     
    for i, prompt in enumerate(prompts):
        if i == 0:
            content = f"Paper Content:\n{paper_content}\n\nTask:\n{prompt}"
        else:
            content = prompt
             
        messages.append({"role": "user", "content": content}) 

        try:
            response = client.chat(messages=messages, model=model_to_use)
            messages.append({"role": "assistant", "content": response})
            
            if i == len(prompts) - 1:
                final_rdf = response
        except Exception as e:
            final_rdf = f"Error during processing: {str(e)}"
            break
    
    final_results.append({
        "LLM_Model": model_to_use,
        "Paper_Code": paper_code,
        "Category": category,
        "Output": final_rdf
    })

    df_output = pd.DataFrame(final_results)
    df_output.to_excel(OUTPUT_FILE, index=False)

NameError: name 'df_subset' is not defined